In [ ]:
"""
Market Sizing Calculator
========================
A McKinsey-style TAM / SAM / SOM modeling tool.
Supports top-down and bottom-up approaches.

Usage:
    python market_sizing.py

Author: Your Name
"""

def format_number(n):
    """Format large numbers into readable strings."""
    if n >= 1_000_000_000:
        return f"${n/1_000_000_000:.2f}B"
    elif n >= 1_000_000:
        return f"${n/1_000_000:.2f}M"
    elif n >= 1_000:
        return f"${n/1_000:.2f}K"
    return f"${n:.2f}"


def top_down_model(config):
    """
    Top-Down Market Sizing
    ----------------------
    Start from total population, filter down to target market.
    Classic McKinsey / consulting approach.
    """
    print("\n" + "="*55)
    print("  TOP-DOWN MARKET SIZING MODEL")
    print("="*55)

    population        = config["total_population"]
    internet_rate     = config["internet_penetration"]       # %
    target_demo_rate  = config["target_demographic_pct"]     # %
    awareness_rate    = config["brand_awareness_pct"]        # %
    conversion_rate   = config["conversion_rate_pct"]        # %
    arpu              = config["avg_revenue_per_user"]       # $

    # Funnel
    internet_users    = population * internet_rate
    target_demo       = internet_users * target_demo_rate
    aware_users       = target_demo * awareness_rate
    customers         = aware_users * conversion_rate
    revenue           = customers * arpu

    print(f"\n  {'Total Population':<35} {population:>15,.0f}")
    print(f"  {'Internet Users ({:.0%})':<35}".format(internet_rate) + f" {internet_users:>15,.0f}")
    print(f"  {'Target Demographic ({:.0%})':<35}".format(target_demo_rate) + f" {target_demo:>15,.0f}")
    print(f"  {'Brand-Aware ({:.0%})':<35}".format(awareness_rate) + f" {aware_users:>15,.0f}")
    print(f"  {'Converted Customers ({:.0%})':<35}".format(conversion_rate) + f" {customers:>15,.0f}")
    print(f"\n  {'ARPU':<35} {format_number(arpu):>15}")
    print(f"  {'━'*50}")
    print(f"  {'ESTIMATED ANNUAL REVENUE (SOM)':<35} {format_number(revenue):>15}")
    print()

    return {
        "total_population": population,
        "internet_users": internet_users,
        "target_demographic": target_demo,
        "aware_users": aware_users,
        "customers": customers,
        "revenue_som": revenue,
    }


def bottom_up_model(config):
    """
    Bottom-Up Market Sizing
    -----------------------
    Build from unit economics and scale up.
    Great for validating top-down results.
    """
    print("\n" + "="*55)
    print("  BOTTOM-UP MARKET SIZING MODEL")
    print("="*55)

    total_businesses  = config["total_addressable_businesses"]
    smb_pct           = config["smb_percentage"]              # %
    willing_to_pay    = config["willing_to_pay_pct"]          # %
    avg_contract      = config["avg_annual_contract_value"]   # $
    market_share_goal = config["realistic_market_share_pct"]  # %

    # Build up
    smb_segment       = total_businesses * smb_pct
    paying_customers  = smb_segment * willing_to_pay
    tam               = paying_customers * avg_contract
    sam               = tam * smb_pct
    som               = sam * market_share_goal

    print(f"\n  {'Total Addressable Businesses':<35} {total_businesses:>15,.0f}")
    print(f"  {'SMB Segment ({:.0%})':<35}".format(smb_pct) + f" {smb_segment:>15,.0f}")
    print(f"  {'Willing to Pay ({:.0%})':<35}".format(willing_to_pay) + f" {paying_customers:>15,.0f}")
    print(f"  {'Avg Annual Contract Value':<35} {format_number(avg_contract):>15}")
    print(f"\n  {'TAM (Total Addressable Market)':<35} {format_number(tam):>15}")
    print(f"  {'SAM (Serviceable Addressable)':<35} {format_number(sam):>15}")
    print(f"  {'SOM (Serviceable Obtainable, {:.0%})':<35}".format(market_share_goal) + f" {format_number(som):>15}")
    print()

    return {"tam": tam, "sam": sam, "som": som}


def sensitivity_analysis(base_customers, base_arpu, label="Revenue"):
    """
    Sensitivity Analysis
    --------------------
    Shows how revenue changes across ±20% swings in key inputs.
    """
    print("\n" + "="*55)
    print("  SENSITIVITY ANALYSIS")
    print("="*55)
    print(f"\n  {label} sensitivity to Customers × ARPU\n")

    arpu_range = [base_arpu * m for m in [0.8, 0.9, 1.0, 1.1, 1.2]]
    cust_range = [base_customers * m for m in [0.8, 0.9, 1.0, 1.1, 1.2]]

    # Header row
    header = f"  {'Customers \\ ARPU':<18}"
    for arpu in arpu_range:
        header += f"  {format_number(arpu):>10}"
    print(header)
    print("  " + "─"*68)

    for customers in cust_range:
        row_label = f"  {customers:>14,.0f}  │"
        for arpu in arpu_range:
            rev = customers * arpu
            row_label += f"  {format_number(rev):>10}"
        print(row_label)
    print()


def run_example():
    """Run a sample market sizing for a B2B SaaS company."""

    print("\n" + "█"*55)
    print("  McKINSEY-STYLE MARKET SIZING CALCULATOR")
    print("  Example: B2B Project Management SaaS — UAE Market")
    print("█"*55)

    # --- Top-Down Config ---
    top_down_config = {
        "total_population":        3_500_000,   # UAE working professionals
        "internet_penetration":    0.98,         # 98% internet penetration
        "target_demographic_pct":  0.30,         # SMB decision makers
        "brand_awareness_pct":     0.40,         # Brand / category awareness
        "conversion_rate_pct":     0.05,         # 5% trial-to-paid
        "avg_revenue_per_user":    1_200,        # $1,200 / year
    }

    td = top_down_model(top_down_config)

    # --- Bottom-Up Config ---
    bottom_up_config = {
        "total_addressable_businesses": 300_000,  # registered SMBs in UAE
        "smb_percentage":               0.70,
        "willing_to_pay_pct":           0.15,
        "avg_annual_contract_value":    4_800,    # $4,800 / year per biz
        "realistic_market_share_pct":   0.05,
    }

    bu = bottom_up_model(bottom_up_config)

    # --- Sensitivity ---
    sensitivity_analysis(
        base_customers = td["customers"],
        base_arpu      = top_down_config["avg_revenue_per_user"],
    )

    # --- Summary ---
    print("="*55)
    print("  EXECUTIVE SUMMARY")
    print("="*55)
    print(f"\n  Top-Down SOM Estimate:    {format_number(td['revenue_som'])}")
    print(f"  Bottom-Up SOM Estimate:   {format_number(bu['som'])}")
    avg = (td["revenue_som"] + bu["som"]) / 2
    print(f"\n  Triangulated Estimate:    {format_number(avg)}")
    print(f"\n  Key Assumption: Market share goal of 5% over 3 years.")
    print(f"  Recommend pressure-testing conversion rate (currently 5%).\n")
    print("="*55 + "\n")


if __name__ == "__main__":
    run_example()



███████████████████████████████████████████████████████
  McKINSEY-STYLE MARKET SIZING CALCULATOR
  Example: B2B Project Management SaaS — UAE Market
███████████████████████████████████████████████████████

  TOP-DOWN MARKET SIZING MODEL

  Total Population                          3,500,000
  Internet Users (98%)                   3,430,000
  Target Demographic (30%)               1,029,000
  Brand-Aware (40%)                        411,600
  Converted Customers (5%)                 20,580

  ARPU                                         $1.20K
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ESTIMATED ANNUAL REVENUE (SOM)              $24.70M


  BOTTOM-UP MARKET SIZING MODEL

  Total Addressable Businesses                300,000
  SMB Segment (70%)                        210,000
  Willing to Pay (15%)                      31,500
  Avg Annual Contract Value                    $4.80K

  TAM (Total Addressable Market)             $151.20M
  SAM (Serviceable Addressable)          

In [ ]:
# 1. Install necessary libraries
!pip install pdfplumber pandas -q

import re
import pandas as pd

# Demo text simulating a consulting report
REPORT_TEXT = """
MARKET UPDATE 2026
Our analysis shows that Revenue grew by 15% reaching $1.2 billion this year.
Current EBITDA is estimated at $300 million.
The firm currently employs 4,500 staff members globally.
CAGR is projected at 8.5% over the next three years.
"""

def quick_scrape(text):
    # Search patterns
    metrics = {
        "Revenue": r"\$[\d,.]+\s*(?:billion|million|B|M)",
        "Growth/CAGR": r"\d+\.?\d*\s*%",
        "Headcount": r"[\d,]+\s*(?:staff|employees|headcount)"
    }

    print("--- EXTRACTED CONSULTING METRICS ---")
    for label, pattern in metrics.items():
        found = re.findall(pattern, text, re.IGNORECASE)
        print(f"{label:12}: {found if found else 'Not found'}")

# Run the scraper
quick_scrape(REPORT_TEXT)

print("\n[Note] To use a real PDF: Upload it to the left sidebar, then use:")
print("import pdfplumber")
print("with pdfplumber.open('your_file.pdf') as pdf: text = pdf.pages[0].extract_text()")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 95.8 MB/s eta 0:00:00
--- EXTRACTED CONSULTING METRICS ---
Revenue     : ['$1.2 billion', '$300 million']
Growth/CAGR : ['15%', '8.5%']
Headcount   : ['4,500 staff']

[Note] To use a real PDF: Upload it to the left sidebar, then use:
import pdfplumber
with pdfplumber.open('your_file.pdf') as pdf: text = pdf.pages[0].extract_text()


In [ ]:
# 1. Setup
!pip install pdfplumber pandas -q

import re
import pandas as pd

# Simulating a sophisticated consulting page
CONSULTING_REPORT = """
CONFIDENTIAL: PROJECT VELOCITY - PHASE 2
Financial Highlights:
In Q4, the company saw a massive surge in Revenue to $4.2B, a 22% increase over the previous fiscal year.
Operating EBITDA stood at $850M with a margin contraction of 200bps.
Total Headcount: 12,400 FTEs.
Market Share: 18.5% (Up from 15% in 2025).
Risks: Inflationary pressure on COGS is rising by 12% annually.
"""

def diagnostic_scrape(text):
    print("="*60)
    print("      DIAGNOSTIC PDF EXTRACTION: PROJECT VELOCITY")
    print("="*60)

    patterns = {
        "Revenue": r"\$[\d,.]+\s*[BM]",
        "Growth": r"\d+\.?\d*\s*%",
        "Headcount": r"[\d,]+\s*(?:FTEs|staff|employees)",
        "EBITDA": r"EBITDA\s*(?:at|is|of)?\s*\$[\d,.]+\s*[BM]?"
    }

    extracted = {}
    for key, p in patterns.items():
        matches = re.findall(p, text, re.IGNORECASE)
        extracted[key] = matches[0] if matches else "N/A"

    # --- Automated "Meat" Analysis ---
    print(f"\n[DATA POINTS]")
    for k, v in extracted.items():
        print(f" > {k:12}: {v}")

    print(f"\n[ANALYST INSIGHTS]")
    # Logic-based analysis
    growth_val = float(re.findall(r"\d+", extracted["Growth"])[0]) if extracted["Growth"] != "N/A" else 0
    if growth_val > 20:
        print(" • PERFORMANCE: High-growth trajectory confirmed (>20% YoY).")

    if "B" in extracted["Revenue"]:
        print(" • SCALE: Entity is operating at a Tier-1 Market capitalization scale.")

    if "COGS" in text:
        print(" • RISK ALERT: Supply chain/COGS mentions suggest margin pressure despite growth.")

diagnostic_scrape(CONSULTING_REPORT)

      DIAGNOSTIC PDF EXTRACTION: PROJECT VELOCITY

[DATA POINTS]
 > Revenue     : $4.2B
 > Growth      : 22%
 > Headcount   : 12,400 FTEs
 > EBITDA      : N/A

[ANALYST INSIGHTS]
 • PERFORMANCE: High-growth trajectory confirmed (>20% YoY).
 • SCALE: Entity is operating at a Tier-1 Market capitalization scale.
 • RISK ALERT: Supply chain/COGS mentions suggest margin pressure despite growth.
